In [1]:
import os
import json
import pandas as pd

from pathlib import Path


# Get GEO_ID from folder name (same pattern as prepare_data.ipynb)
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))


description_path = Path('description.json')
preprocessed_csv_path = Path(f"{GEO_ID}_preprocessed.csv")

if not description_path.exists():
    raise FileNotFoundError(
        "description.json not found. Please run 'prepare_data.ipynb' first to generate it."
    )

if not preprocessed_csv_path.exists():
    raise FileNotFoundError(
        f"{preprocessed_csv_path} not found. Please run 'preprocess_data.ipynb' first to generate it."
    )

with open(description_path) as f:
    summary_data = json.load(f)

df = pd.read_csv(preprocessed_csv_path)

print("Data contains", df.shape[0], "samples and", df.shape[1], "columns.")
print("First few rows of the data:")
df.head()


Data contains 27 samples and 14875 columns.
First few rows of the data:


,time to last followup,survstatus,sample_title,diagnosis,tissue,Sex,age,Stage,cell type,differentiation,...,COX2,ATP8,ATP6,COX3,ND3,ND4L,ND4,ND5,ND6,CYTB
0,8.59,0,Lu1031A,Lung Adenocarcinoma,Lung,Female,57,IA,AD,Well,...,7.609257,6.404654,7.794737,7.763609,6.390111,6.155748,8.336788,7.752943,7.057440,7.787434
1,7.54,0,Lu1050A,Lung Adenocarcinoma,Lung,Male,58,IB,AD,Well,...,8.960444,7.783257,9.031700,9.397762,7.223517,7.342558,9.427546,8.544923,7.714072,9.010870
2,3.28,1,Lu1068A,Lung Adenocarcinoma,Lung,Female,57,IB,AD,Mod,...,8.411483,7.260510,8.586719,8.632693,6.731376,7.070203,9.103918,8.805666,8.002787,8.480207
3,11.96,0,Lu106A,Lung Adenocarcinoma,Lung,Male,63,IB,AD,Well,...,7.782319,6.535627,7.859483,8.173400,6.839753,6.052102,8.200441,7.069845,6.470527,7.914763
4,5.68,1,Lu113A,Lung Adenocarcinoma,Lung,Female,89,IA,AD,Well,...,8.120021,6.914451,8.065773,8.618442,6.605339,6.447653,8.449232,7.963453,7.131778,7.998035


In [2]:
n_stats_written = 0

for idx, endpoint in enumerate(summary_data['survival-endpoints']):
    time_var = endpoint['time_var']['var_name']
    event_var = endpoint['event_var']['var_name']

    if time_var not in df.columns or event_var not in df.columns:
        abbrv = endpoint.get('abbrv', f'endpoint[{idx}]')
        print(
            f"Skipping stats for {abbrv}: columns not in preprocessed table "
            f"({time_var!r}, {event_var!r})."
        )
        endpoint.pop('stats', None)
        continue

    complete_mask = df[[time_var, event_var]].notnull().all(axis=1)
    n_complete = int(complete_mask.sum())
    n_incomplete = int(len(df) - n_complete)

    # Among samples with complete time + event data
    censored_count = int((complete_mask & (df[event_var] == 0.0)).sum())
    event_count = int((complete_mask & (df[event_var] == 1.0)).sum())

    censored_ratio = censored_count / n_complete

    print(f"Number of samples with incomplete data for {time_var} or {event_var}: {n_incomplete}")
    print(f"Number of samples with complete data for both variables: {n_complete}")
    print(f"Number of censored samples (event_var == 0.0): {censored_count}")
    print(f"Number of events (event_var == 1.0): {event_count}")
    pct = 100.0 * censored_ratio
    print(
        f"Censored ratio (among complete): {censored_count} / {n_complete} = "
        f"{censored_ratio:.3f} ({pct:.1f}%)"
    )

    # Refresh only this block on each run; leave all other endpoint fields unchanged
    endpoint['stats'] = {
        'n_incomplete': n_incomplete,
        'n_complete': n_complete,
        'n_censored': censored_count,
        'n_events': event_count,
        'censored_ratio': round(censored_ratio, 2),
    }
    n_stats_written += 1

with open(description_path, 'w', encoding='utf-8') as f:
    json.dump(summary_data, f, indent=4, ensure_ascii=False)
    f.write('\n')

print(
    f"Updated stats for {n_stats_written} of {len(summary_data['survival-endpoints'])} endpoint(s); "
    f"wrote {description_path.resolve()}"
)


Number of samples with incomplete data for time to last followup or survstatus: 0
Number of samples with complete data for both variables: 27
Number of censored samples (event_var == 0.0): 20
Number of events (event_var == 1.0): 7
Censored ratio (among complete): 20 / 27 = 0.741 (74.1%)
Updated stats for 1 of 1 endpoint(s); wrote /Users/jakakokosar/dev/phd/survival-rnaseq-data/datasets/geo/GSE87340/description.json
